# Exercises (Student) - MCP Client with LLM

In [1]:
!pip install -q mcp nest_asyncio requests

In [2]:
import os
from pathlib import Path
MCP_HTTP_TOKEN = os.getenv("MCP_HTTP_TOKEN", "devtoken123")
USE_REAL_LLM = False  # flip True if GITHUB_TOKEN is set

In [3]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

In [4]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("DemoServer")

# ✅ Decorator registers `add` as an MCP tool
@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b  # ✅ returns the sum

# Optional bonus: multiply tool
@mcp.tool()
def multiply(a: int, b: int) -> int:
    "Multiply two numbers."
    return a * b

# ✅ Decorator registers `greet` as an MCP tool
@mcp.tool()
def greet(name: str) -> str:
    "Return a greeting string."
    return f"Hello, {name}!"  # ✅ returns a greeting message

if __name__ == "__main__":
    mcp.run()

Writing server.py


## Exercise 1 (provide answer)

### Why is STDIO transport simpler for local MCP dev compared to HTTP?

**STDIO transport** (Standard Input/Output) is simpler for local MCP development for several reasons:

1. **No networking setup required**: STDIO communicates directly through the process's stdin/stdout pipes. There is no need to bind a port, configure a host, handle TLS/SSL certificates, or deal with firewall rules.

2. **No authentication overhead**: HTTP servers typically require tokens, API keys, or session management (as shown by `MCP_HTTP_TOKEN` in this notebook). With STDIO, the client simply spawns the server as a child process — OS-level process isolation provides the security boundary.

3. **Zero configuration**: The client just needs the command to launch the server (`python server.py`). With HTTP, both sides must agree on a host, port, and protocol before any message is exchanged.

4. **Process lifecycle is automatic**: When the client exits, the child process is cleaned up. With HTTP, the server is a long-running daemon that must be started, stopped, and managed separately.

5. **No serialization ambiguity**: Messages flow as newline-delimited JSON over a single pipe — no HTTP headers, status codes, or connection pooling to reason about.

In short: STDIO collapses client + server into a single `subprocess` call, making it ideal for rapid local iteration before graduating to HTTP for multi-client or remote deployments.

## Exercise 2

In [5]:
import os
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()

async def ex2_connect():
    # ✅ StdioServerParameters: command = python interpreter, args = server script
    params = StdioServerParameters(command="python", args=["server.py"])
    errlog = open(os.devnull, "w")  # Fix: Jupyter replaces sys.stderr with a non-file stream
    async with stdio_client(params, errlog=errlog) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()

In [6]:
await ex2_connect()
print("Exercise 2: OK (connected and initialized)")

Exercise 2: OK (connected and initialized)


## Exercise 3

In [7]:
import os
async def ex3_list():
    # ✅ Same StdioServerParameters pattern
    params = StdioServerParameters(command="python", args=["server.py"])
    errlog = open(os.devnull, "w")  # Fix: Jupyter replaces sys.stderr with a non-file stream
    async with stdio_client(params, errlog=errlog) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            # ✅ list_resources() returns the server's declared resources
            resources = await session.list_resources()
            print("RESOURCES:", resources)
            tools = await session.list_tools()
            for t in tools.tools:
                print(t.name, t.inputSchema.get("properties", {}))

In [8]:
await ex3_list()

RESOURCES: meta=None nextCursor=None resources=[]
add {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
multiply {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
greet {'name': {'title': 'Name', 'type': 'string'}}


## Exercise 4

### How does the conversion to LLM tool happen in MCP server code?

When FastMCP registers a function with `@mcp.tool()`, it introspects the function's **type annotations** and **docstring** to automatically generate a JSON Schema (`inputSchema`). For example:

```python
@mcp.tool()
def add(a: int, b: int) -> int:
    "Add two numbers."
    ...
```

FastMCP converts this into an MCP `Tool` object with:
- `name`: `"add"`
- `description`: `"Add two numbers."` (from the docstring)
- `inputSchema`: `{"type": "object", "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}}, "required": ["a", "b"]}`

The `convert_to_llm_tool()` function below then **re-wraps** this MCP schema into the OpenAI function-calling format that LLMs expect — specifically the `{"type": "function", "function": {...}}` envelope. This translation is the bridge that lets an LLM reason about which MCP tools to invoke and with what arguments.

In [9]:
def convert_to_llm_tool(tool):
    """
    Converts an MCP Tool object into the OpenAI function-calling spec format.

    MCP Tool has:
      - tool.name        : str
      - tool.description : str | None
      - tool.inputSchema : dict (JSON Schema object)

    LLM tools expect:
      {"type": "function", "function": {"name", "description", "parameters": {JSON Schema}}}
    """
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": tool.description or "mcp tool",
            "parameters": {
                "type": "object",
                "properties": tool.inputSchema.get("properties", {}),
                "required": tool.inputSchema.get("required", []),
            },
        },
    }

## Exercise 5

**Plan & execute:** Use stub (or real) LLM to propose `tool_calls`, then execute them and print results for a prompt like "Add 2 to 20."

In [10]:
import asyncio
import json
import re
import nest_asyncio
from typing import Any, Dict, List
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

nest_asyncio.apply()


def stub_plan(prompt: str, functions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Minimal keyword-based stub planner — no LLM token needed.
    Parses the prompt to identify which tool to call and extracts integer arguments.
    """
    prompt_lower = prompt.lower()
    numbers = [int(n) for n in re.findall(r"\d+", prompt)]

    if "multiply" in prompt_lower and len(numbers) >= 2:
        return [{"name": "multiply", "args": {"a": numbers[0], "b": numbers[1]}}]
    elif "add" in prompt_lower and len(numbers) >= 2:
        return [{"name": "add", "args": {"a": numbers[0], "b": numbers[1]}}]
    elif "greet" in prompt_lower or "hello" in prompt_lower:
        # Extract a name after "greet" or "hello"
        match = re.search(r"(?:greet|hello)\s+(\w+)", prompt_lower)
        name = match.group(1).capitalize() if match else "World"
        return [{"name": "greet", "args": {"name": name}}]
    else:
        print(f"[stub_plan] Could not parse prompt: '{prompt}'")
        return []


def call_llm(prompt: str, functions: List[Dict[str, Any]], use_real: bool = False):
    if not use_real:
        return stub_plan(prompt, functions)
    token = os.getenv("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Set GITHUB_TOKEN or use stub planner.")
    from azure.ai.inference import ChatCompletionsClient
    from azure.core.credentials import AzureKeyCredential
    client = ChatCompletionsClient("https://models.inference.ai.azure.com", AzureKeyCredential(token))
    resp = client.complete(
        model="gpt-4o",
        messages=[{"role": "system", "content": "Plan MCP tool calls."},{"role": "user", "content": prompt}],
        tools=functions,
        temperature=0,
        max_tokens=400,
    )
    calls = []
    msg = resp.choices[0].message
    for tc in msg.tool_calls or []:
        args = tc.function.arguments
        args_json = json.loads(args) if isinstance(args, str) else args
        calls.append({"name": tc.function.name, "args": args_json})
    return calls

In [11]:
import os
async def ex5_run(prompt: str = "Add 2 to 20"):
    # ✅ StdioServerParameters to launch server.py
    params = StdioServerParameters(command="python", args=["server.py"])
    errlog = open(os.devnull, "w")  # Fix: Jupyter replaces sys.stderr with a non-file stream
    async with stdio_client(params, errlog=errlog) as (r, w):
        async with ClientSession(r, w) as session:
            await session.initialize()
            tools = await session.list_tools()
            # ✅ Convert each MCP Tool into the OpenAI function-calling format
            functions = [convert_to_llm_tool(t) for t in tools.tools]
            # ✅ Ask the (stub) LLM which tool(s) to call
            calls = call_llm(prompt, functions, use_real=USE_REAL_LLM)
            print("tool_calls:", calls)
            for call in calls:
                result = await session.call_tool(call["name"], arguments=call["args"])
                print("result:", [getattr(c, "text", str(c)) for c in result.content])

In [12]:
await ex5_run("Add 2 to 20")

tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result: ['22']


## Optional - add multiply(a, b) and rerun

The `multiply` tool is already registered in `server.py` above with `@mcp.tool()`.  
Just rerun the cell below — the planner will discover it automatically via `list_tools()`.

In [13]:
# Optional: test multiply and greet tools
await ex5_run("Multiply 6 by 7")

tool_calls: [{'name': 'multiply', 'args': {'a': 6, 'b': 7}}]
result: ['42']


In [14]:
await ex5_run("Greet Alice")

tool_calls: [{'name': 'greet', 'args': {'name': 'Alice'}}]
result: ['Hello, Alice!']
